# Week 09 · Reliability through controlled experiments
**Machine Learning in Engineering Science — NCKU + LAiMM**

Monday 2 November 2026, 14:00–17:00 (Taiwan). Three experiments, CPU only; no GPU, key or paid AI tool. This is formative class practice, not new graded Homework. **Midterm: written and oral examinations only; no midterm project.**

**Worked alternative:** contains example code and responses. Read it after attempting your own prediction.

## Route and timing
H1: 25 min mechanisms + 35 min Experiment A. H2: 20 min mechanisms + 25 min B + 10 min break + 5 min buffer. H3: 15 min explanation/summary + 40 min C + 5 min buffer. Total: 60 explanation, 100 practice/discussion, 20 break/buffer.

Before each run: write your prediction. After it: cite one measured value or plot and explain the mechanism. If your runtime resets, run from the top. In Colab choose Python 3 CPU; the standard environment includes NumPy, Matplotlib and scikit-learn. If a local import fails, install those three in your own environment and restart. No installation or download is required by this notebook.

## Setup — predeclare a fair comparison
Independent synthetic inputs: $x_j\sim N(0,1)$; $y=2x_0+x_1^2+\epsilon$, noise SD 0.2. Three raw inputs are dimensionless; the output is in synthetic response units. These are neither stock data nor equipment measurements. Split exactly 350 train / 75 validation / 75 test. Fit every scaler on train only. Two candidates and validation MAE selection (ties favor linear) are declared before any test score. The two-model test comparison is a predeclared diagnosis, not permission to reselect.

Quick review (≤5 min): residual = prediction − observation; MAE averages absolute residuals, RMSE square-roots mean squared residuals. Both have response units. Neither a single average nor a single residual plot separates all error sources.

In [ ]:
from pathlib import Path
import json, hashlib, zipfile, platform, sys, inspect, copy
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
OUT=Path('week09-artifacts'); OUT.mkdir(exist_ok=True)
responses={}
FIGURES_THIS_RUN=[]
plt.rcParams.update({'figure.figsize':(10,6),'font.size':13,'axes.spines.top':False,'axes.spines.right':False})
def truth(X): return 2*X[:,0]+X[:,1]**2
def sample(seed,n=500):
    rng=np.random.default_rng(seed); X=rng.normal(size=(n,3)); noise=rng.normal(0,.2,n)
    return X,truth(X)+noise,noise
def square(X): return np.column_stack([X,X[:,1]**2])
def model(kind):
    return make_pipeline(FunctionTransformer(square if kind=='quadratic' else None,validate=True),StandardScaler(),Ridge(alpha=1,solver='svd'))
def metrics(y,p): return {'MAE':float(mean_absolute_error(y,p)),'RMSE':float(np.sqrt(mean_squared_error(y,p)))}
def savefig(name):
    plt.tight_layout(); plt.savefig(OUT/(name+'.png'),dpi=160); plt.show(); plt.close(); FIGURES_THIS_RUN.append(name+'.png')
X,y,noise=sample(2026)
train,held=train_test_split(np.arange(500),test_size=.3,random_state=2026)
val,test=train_test_split(held,test_size=.5,random_state=2027)
train,val,test=map(np.sort,(train,val,test))
assert len(train)==350 and len(val)==len(test)==75
assert not(set(train)&set(val) or set(train)&set(test) or set(val)&set(test))
models={k:model(k).fit(X[train],y[train]) for k in ['linear','quadratic']}
validation={k:metrics(y[val],m.predict(X[val])) for k,m in models.items()}
selected=min(models,key=lambda k:(validation[k]['MAE'],k!='linear'))
print('Validation only:',validation,'Frozen choice:',selected)
# Selection is complete before the predeclared test comparison is computed.
testing={k:metrics(y[test],m.predict(X[test])) for k,m in models.items()}
print('Predeclared test diagnosis:',testing)
split=np.full(500,'train',dtype='<U5'); split[val]='val';split[test]='test'
with (OUT/'data.csv').open('w') as f:
    f.write('row_id,x0,x1,x2,y,split\n')
    for i in range(500):f.write(','.join([str(i),*(format(z,'.17g') for z in X[i]),format(y[i],'.17g'),split[i]])+'\n')
results={'validation':validation,'test':testing,'selected':selected,'seed':2026,'noise_variance':.04}
print('Setup complete. Independent synthetic rows; CPU only; no API key or GPU.')

## Experiment A — Separate model form, noise and refit variation · 35 minutes
Predict which error component should decrease when we increase training rows while keeping the feature family fixed. Will collecting more data repair a missing square? Write first, without opening the worked answers.

Then run the shared-scale residual diagnosis and 20 independently refitted models at each sample size. Inspect the mean and band, not just one fit. The fixed probe labels are noise-free: this isolates model error. Explain why scatter around a curved residual is insufficient to measure training-sample variability.

In [ ]:
responses['A_prediction'] = 'Variance can shrink while a missing shape persists.'

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4.8),sharex=True,sharey=True)
for a,(k,m) in zip(ax,models.items()):
    a.scatter(X[test,1],m.predict(X[test])-y[test],s=27,alpha=.7)
    a.axhline(0,c='black',lw=1);a.set(title=k,xlabel='x1',ylabel='prediction - observation')
savefig('residual-shared')
z=np.linspace(-2.8,2.8,120)
plt.plot(z,1-z*z,label='Population affine residual mean: 1 - x1²')
plt.scatter(X[test,1],models['linear'].predict(X[test])-y[test],s=25,label='Finite Ridge residuals')
plt.axhline(0,c='black',lw=1);plt.xlabel('x1');plt.ylabel('residual');plt.legend();savefig('residual-mechanism')
# Exact population statement requires independent standard-normal inputs, no regularization,
# and a population least-squares projection. Finite regularized Ridge is not that exact line.
print('Population affine projection: 1 + 2*x0. Noise variance .04; population MSE 2.04.')
print('Correct quadratic oracle MSE .04. This is an expectation, not an exact sample score.')

In [ ]:
# The probe is fixed and noise-free: it isolates training-sample variability.
probe=np.random.default_rng(3001).normal(size=(240,3)); target=truth(probe)
sizes=[40,160,640]; repeats=20; decomposition=[]; predictions={}
for size in sizes:
    for kind in models:
        P=[]
        for r in range(repeats):
            # Prefixes make the two model families and sample sizes paired within a repetition.
            xr,yr,_=sample(4000+r,640)
            fit=model(kind).fit(xr[:size],yr[:size]);P.append(fit.predict(probe))
        P=np.array(P);mean=P.mean(axis=0)
        bias2=float(np.mean((mean-target)**2));variance=float(np.mean((P-mean)**2))
        signal_mse=float(np.mean((P-target)**2))
        assert np.isclose(signal_mse,bias2+variance,atol=1e-12)
        decomposition.append({'kind':kind,'n':size,'empirical_bias2':bias2,'refit_variance':variance,'signal_mse':signal_mse,'noise_averaged_mse':signal_mse+.04})
        predictions[kind,size]=P
print(json.dumps(decomposition,indent=2))
fig,axes=plt.subplots(1,2,figsize=(11,4.8),sharey=True)
for ax,kind in zip(axes,models):
    rows=[r for r in decomposition if r['kind']==kind]
    ax.plot(sizes,[r['empirical_bias2'] for r in rows],'o-',label='empirical bias²')
    ax.plot(sizes,[r['refit_variance'] for r in rows],'s-',label='refit variance')
    ax.axhline(.04,c='grey',ls=':',label='known noise variance')
    ax.set(xscale='log',xlabel='training rows',ylabel='squared response units',title=kind);ax.legend()
savefig('bias-variance')
# Slice bands across independently refitted models, not confidence intervals for future labels.
sliceX=np.column_stack([np.zeros(100),np.linspace(-2.5,2.5,100),np.zeros(100)])
fig,axes=plt.subplots(1,2,figsize=(11,4.8),sharey=True)
for ax,kind in zip(axes,models):
    P=np.array([model(kind).fit(*sample(5000+r,40)[:2]).predict(sliceX) for r in range(repeats)])
    mean=P.mean(0);sd=P.std(0)
    ax.plot(sliceX[:,1],truth(sliceX),'k--',label='noise-free truth')
    ax.plot(sliceX[:,1],mean,label='mean of 20 fits');ax.fill_between(sliceX[:,1],mean-sd,mean+sd,alpha=.25,label='±1 refit SD')
    ax.set(title=kind,xlabel='x1; x0=x2=0',ylabel='predicted response');ax.legend(fontsize=10)
savefig('refit-bands')
results['decomposition']=decomposition

### Interpret the mechanism after execution
For population unregularized least squares under the stated independent Gaussian inputs, the best affine projection is $1+2x_0$. Thus $E[e\mid x_1]=1-x_1^2$. This is a population statement, not the exact finite Ridge fit. Its population MSE is $Var(x_1^2)+Var(\epsilon)=2+0.04=2.04$; the correct quadratic oracle has MSE 0.04.

The repeated-fit array obeys an exact **finite empirical** identity: mean squared error to the known signal = squared error of the mean predictor + mean squared deviation around that mean (variance uses `ddof=0`). Adding 0.04 gives the noise-averaged expected component; it need not equal a single noisy test score. Twenty refits provide a teaching estimate, not a guarantee that all curves are monotone.

In [ ]:
responses['A_explanation'] = 'Compare the curved linear mean with the quadratic mean and refit bands. More rows cannot add the missing square. Noise affects training fits and future observations separately.'

## Experiment B — Change the inputs, then change the rule · 25 minutes
Predict the effect of widening x1 on each frozen model. Then predict the effect of reversing the x0 relationship while keeping X unchanged. State what stays fixed in each experiment.

Run the paired stress sweep: same base rows, same noise, same target rule, only x1 scale changes. Then run the separate target-rule change: exact same test inputs and frozen predictions; only labels change. Compare the mechanisms, not only the direction of MAE.

In [ ]:
responses['B_prediction'] = 'A frozen model can react differently to a changed input mix and a changed target rule.'

In [ ]:
# A predeclared diagnostic stress grid; never used to select or refit the frozen model.
baseX,baseY,baseNoise=sample(2126,150); stress=[]
for scale in [1.,1.5,2.5]:
    changed=baseX.copy();changed[:,1]*=scale
    changedY=truth(changed)+baseNoise
    for kind,m in models.items():stress.append({'kind':kind,'scale':scale,**metrics(changedY,m.predict(changed))})
for kind in models:
    rows=[r for r in stress if r['kind']==kind]
    plt.plot([r['scale'] for r in rows],[r['MAE'] for r in rows],'o-',label=kind)
plt.xlabel('x1 input scale (same base rows, same noise)');plt.ylabel('MAE');plt.legend();savefig('input-shift')
# Changed target relationship: same test X, same noise and exact same frozen predictions.
changedY=-2*X[test,0]+X[test,1]**2+noise[test]
concept={k:metrics(changedY,m.predict(X[test])) for k,m in models.items()}
plt.bar(np.arange(2)-.18,[testing[k]['MAE'] for k in models],.36,label='original rule')
plt.bar(np.arange(2)+.18,[concept[k]['MAE'] for k in models],.36,label='x0 coefficient reversed')
plt.xticks(range(2),list(models));plt.ylabel('MAE');plt.legend();savefig('target-shift')
results.update(stress=stress,concept=concept)
print('Input stress:',stress,'Changed relationship:',concept)
print('Gaussian support is still unbounded. This is a changed distribution, not new mathematical support.')

In [ ]:
responses['B_explanation'] = 'The stress sweep preserves the rule and paired noise; the reversed coefficient changes the conditional relationship. Neither mechanism is established by input monitoring alone.'

## Experiment C — Make and defend your own code change · 40 minutes
Before running: choose ONE X-only feature modification and write your hypothesis, expected outcome and held-fixed conditions. You must edit `STUDENT_SOURCE` itself. A different random seed or polished explanation alone does not complete this experiment.

The function receives X only; preserve the first three columns and optionally append features. Avoid labels, row IDs and split membership. Fit its scaler and Ridge on the original training rows. Preserve the selected baseline. Freeze the code and plan before opening the new audit outcomes. A failed improvement is a valid result if you can explain it. Do not repeatedly optimize against the audit set; disclose any such reuse as exploratory.

In [ ]:
responses['C_prediction'] = 'I predict that my proposed basis change will affect representation error; other conditions stay fixed.'

In [ ]:
# EDIT THIS FUNCTION BODY. The identity is deliberately an unimplemented starting point.
# Keep raw columns in their original order, optionally append new X-only features.
STUDENT_SOURCE = """def student_features(X):
    return np.column_stack([X, X[:,1]**2])
"""
exec(STUDENT_SOURCE)
# Write these BEFORE freeze and audit. One or two precise sentences per field.
plan={
 'hypothesis':'Append the known square to repair affine misspecification', 'expected_outcome':'Match the quadratic reference within numerical tolerance', 'held_fixed':'Same train rows, alpha, scaler, validation selection and audit rows',
 'target':'Synthetic response y', 'split':'350/75/75 original split; a new 240-row audit after freeze', 'baseline':'Validation-selected predeclared reference, preserved', 'evaluation':'Validation MAE before freeze; one held-out audit comparison', 'limitations':'Known synthetic mechanism; no causal or physical validation', 'AI_disclosure':'Instructor worked example; not an independent student submission'
}

In [ ]:
probecheck=X[train[:12]].copy(); before=probecheck.copy()
a=np.asarray(student_features(probecheck));b=np.asarray(student_features(probecheck))
assert np.array_equal(probecheck,before),'Feature code changed its input in place.'
assert a.ndim==2 and a.shape[0]==len(probecheck) and a.shape[1]>=3
assert np.all(np.isfinite(a)) and np.array_equal(a,b),'Features must be finite and deterministic.'
assert np.array_equal(a[:,:3],before),'Keep original raw columns first.'
implemented=not(a.shape==before.shape and np.array_equal(a,before))
# A recorded declaration is a teaching contract, not a security barrier against deliberate re-running.
frozen={'source':STUDENT_SOURCE,'plan':copy.deepcopy(plan),'selected_baseline':selected,'train_ids':train.tolist(),'val_ids':val.tolist()}
frozen['sha256']=hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest()
(OUT/'student-freeze.json').write_text(json.dumps(frozen,indent=2))
student_model=None
if implemented:
    student_model=make_pipeline(FunctionTransformer(student_features,validate=True),StandardScaler(),Ridge(alpha=1,solver='svd')).fit(X[train],y[train])
    student_val=metrics(y[val],student_model.predict(X[val]));print('Student validation:',student_val)
else: print('DRAFT: identity feature function remains unchanged. Edit the function body before claiming completion.')
print('Frozen proposal:',frozen['sha256'])

In [ ]:
# Run only AFTER freezing your proposal. Do not tune repeatedly against these audit outcomes.
current=hashlib.sha256(json.dumps({k:v for k,v in frozen.items() if k!='sha256'},sort_keys=True).encode()).hexdigest()
assert current==frozen['sha256'] and STUDENT_SOURCE==frozen['source'] and plan==frozen['plan'],'Refreeze a changed proposal; disclose a repeated audit as exploratory.'
auditX,auditY,_=sample(92026,240)
audit={'baseline':metrics(auditY,models[selected].predict(auditX))}
if student_model is not None:
    audit['student']=metrics(auditY,student_model.predict(auditX))
    plt.scatter(auditY,models[selected].predict(auditX),label='preserved baseline',alpha=.6)
    plt.scatter(auditY,student_model.predict(auditX),label='student modification',alpha=.6)
    plt.plot([auditY.min(),auditY.max()],[auditY.min(),auditY.max()],'k--');plt.xlabel('observed audit response');plt.ylabel('prediction');plt.legend();savefig('student-audit')
print(audit)
audit_vectors={'X':auditX,'y':auditY,'baseline':models[selected].predict(auditX)}
if student_model is not None:audit_vectors['student']=student_model.predict(auditX)
np.savez(OUT/'student-audit-vectors.npz',**audit_vectors)
results['audit_seed']=92026
results['student']={'implemented':implemented,'freeze_sha256':frozen['sha256'],'audit':audit,'status':'draft until code and explanations are reviewed'}

In [ ]:
responses['C_explanation'] = 'The worked square feature matches the predeclared quadratic pipeline. This is a positive control, not a novel discovery; an honest unsuccessful change can also be defensible.'

## Exit and evidence
A new engineering case has a stable input histogram, yet error increases. Can you conclude there is no distribution problem? Explain what labels, timing, residuals and repeated refits could distinguish. Then defend **target, split, baseline, evaluation, limitations** without reading AI-generated prose.

Run export. `draft` is expected until the function and all response/plan fields are complete. `ready for peer review` means presence, never correctness. Open the ZIP and check its figures, saved code, frozen declaration, data identity and metrics. Save your edited `.ipynb` separately. If an assertion fails, read it before rerunning; check shapes, input mutation, row order and whether you changed your frozen plan. CSV readback uses the same installed environment; Week 10 extends portability.

In [ ]:
# Export only artifacts generated by this run; leftover files are excluded.
results['responses']=responses
status='ready for peer review' if implemented and all(str(v).strip() for v in plan.values()) and all(str(v).strip() for v in responses.values()) else 'draft'
results['status']=status
(OUT/'results.json').write_text(json.dumps(results,indent=2))
(OUT/'student_features.py').write_text(STUDENT_SOURCE)
(OUT/'versions.json').write_text(json.dumps({'python':platform.python_version(),'numpy':np.__version__,'sklearn':sklearn.__version__},indent=2))
np.savez(OUT/'predictions.npz',test_ids=test,y=y[test],**{k:m.predict(X[test]) for k,m in models.items()})
# Same-environment CSV readback, with exact row alignment.
rows=np.genfromtxt(OUT/'data.csv',delimiter=',',names=True,dtype=None,encoding='utf8')
rx=np.column_stack([rows[k] for k in ['x0','x1','x2']]);ry=rows['y']
replayed=model(selected).fit(rx[rows['split']=='train'],ry[rows['split']=='train'])
assert np.allclose(replayed.predict(rx[rows['split']=='test']),models[selected].predict(X[test]),rtol=1e-10,atol=1e-10)
plots=set(FIGURES_THIS_RUN)
if not implemented:plots.discard('student-audit.png')
required_plots={'residual-shared.png','residual-mechanism.png','bias-variance.png','refit-bands.png','input-shift.png','target-shift.png'}
assert required_plots<=plots,'Run Setup and Experiments A and B before exporting.'
if implemented:assert 'student-audit.png' in plots,'Run the modified feature, freeze and audit cells before exporting.'
# Self-contained report: all charts are embedded, with no JavaScript or remote assets.
# It remains usable if the browser cannot load Colab rich-output components.
import base64, html as html_escape
parts=['<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Week09 experiment evidence</title><style>body{max-width:1100px;margin:auto;padding:25px;font:18px/1.6 system-ui;color:#12233f;background:#faf8f3}img{max-width:100%;height:auto}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#fff;padding:18px}figure{margin:35px 0}</style><h1>Week09: controlled experiment evidence</h1>']
parts.append('<p>NCKU + LAiMM · Machine Learning in Engineering Science</p><p>Status: '+html_escape.escape(status)+'. Synthetic class practice; same-environment readback is not physical or independent-environment validation.</p>')
parts.append('<h2>Metrics and explanations</h2><pre>'+html_escape.escape(json.dumps(results,indent=2))+'</pre>')
parts.append('<h2>Frozen feature source</h2><pre>'+html_escape.escape(STUDENT_SOURCE)+'</pre><h2>Declared plan</h2><pre>'+html_escape.escape(json.dumps(plan,indent=2))+'</pre>')
for name in sorted(plots):
    encoded=base64.b64encode((OUT/name).read_bytes()).decode('ascii')
    parts.append('<figure><h2>'+html_escape.escape(name)+'</h2><img alt="'+html_escape.escape(name)+'" src="data:image/png;base64,'+encoded+'"></figure>')
(OUT/'report.html').write_text(''.join(parts)+'</html>',encoding='utf-8')
export_names=plots|{'data.csv','student-freeze.json','student-audit-vectors.npz','predictions.npz','versions.json','results.json','student_features.py','report.html'}
assert all((OUT/name).is_file() for name in export_names)
manifest={'status':status,'scope':'Same-environment CSV readback; charts and offline report generated by this run only.','files':{name:hashlib.sha256((OUT/name).read_bytes()).hexdigest() for name in sorted(export_names)}}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
with zipfile.ZipFile('week09-evidence.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in sorted(export_names|{'manifest.json'}):z.write(OUT/name,name)
print('Export:',status,'→ week09-evidence.zip')
print('Colab: Files sidebar → week09-evidence.zip → right-click → Download.')
print('Unzip locally and open report.html to inspect every chart offline. Save your edited notebook separately.')

## Colab downloads and chart fallback
After export, open the Files sidebar, right-click **week09-evidence.zip**, and choose **Download**. Unzip it on your computer and open **report.html**. The report embeds all charts, metrics, explanations and the feature source; it needs no network or JavaScript. Save your edited notebook separately using File → Download → Download .ipynb.

If Colab says it cannot load JavaScript needed for output, first save and reload the notebook. This is a browser/output-component problem, not proof of a Python failure. Inspect the exported report and PNG files while checking the browser environment. Do not keep rerunning or changing the model merely because a plot is not visible. The export includes only figures generated in the current run, so stale files are excluded. Run from Setup after a runtime reset.

## Summary
A representation can be systematically wrong even with abundant data. Refit variance requires independent retraining. Input shift and changed relationships require separate controls. One defensible code change is more useful than an unexplained score.

References: [scikit-learn bias–variance experiment](https://scikit-learn.org/stable/auto_examples/ensemble/plot_bias_variance.html), [leakage-safe pipelines](https://scikit-learn.org/stable/common_pitfalls.html), [Google ML monitoring](https://developers.google.com/machine-learning/crash-course/production-ml-systems/monitoring).